# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from joblib import dump, load
from sklearn.pipeline import Pipeline
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.options.display.max_rows = 10

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        return self

    def transform(self, X):
        X['hour'] = X['timestamp'].dt.hour
        X['dayofweek'] = X['timestamp'].dt.weekday
        del X['timestamp']
        return pd.DataFrame(X)

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_name):
        super().__init__()
        self.target_name = target_name

    def fit(self, X, y = None):
        return self

    def transform(self, X):
        try:
            y = X[self.target_name]
            X = X.drop(self.target_name, axis=1)
            categorical_data = [feature for feature in X if type(X[feature].iloc[0]) == str and feature != self.target_name]
            numeric_data = set(list(X.columns)) - set(categorical_data)
            numeric_data = X[list(numeric_data)]

            enc = OneHotEncoder(handle_unknown='ignore')
            enc.fit(X[categorical_data])
            categorical_data = enc.transform(X[categorical_data]).toarray()
            categorical_inames = [y for x in enc.categories_ for y in x]
            categorical_data = pd.DataFrame(categorical_data, columns=categorical_inames)

        except Exception as e:
            print(f'error: {e}')
            
        return pd.concat([categorical_data, numeric_data], axis=1), y

In [4]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    def __init__(self, y):
        super().__init__()
        self.y = y

    def fit(self, X, y = None):
        return self

    def transform(self, X):
        X_train, X_test, y_train, y_test = train_test_split(X, self.y, test_size=0.2, random_state=21, stratify=self.y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.772727
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.801484
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.855288
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection():
    def __init__(self, grids, grids_dict):
        self.grids = grids
        self.grids_dict = grids_dict
        self.best_model = []

    def choose(self):
        best_model = []

        for model_num, model_class in enumerate(self.grids):
            
            best: tuple = ('', 0, 0)
            print(f'Estimator: {self.grids_dict[model_num]}')

            for model_params in tqdm(ParameterGrid(model_class.param_grid), delay=1):
               
                model = model_class.estimator(**model_params)
                model.fit(X_train, y_train)

                model_score_train = round(np.mean(cross_val_score(model, X_train, y_train, cv=2, n_jobs=-1)), 3)
                model_score_valid = round(accuracy_score(model.predict(X_valid), y_valid), 3)

                if model_score_train > best[1]:
                    best = (model_params, model_score_train, model_score_valid, model_class, self.grids_dict[model_num])

            best_model.append(best)
            print(f'Best params {best[0]}')
            print(f'Best training accuracy: {best[1]}')
            print(f'Validation set accuracy score for best params: {best[2]}\n')
        
        
        self.best_model = best_model
        best_model.sort(key=lambda x: x[1], reverse=True)
        print(f'Classifier with best validation set accuracty {best_model[0][-1]}')

        return sorted(self.best_model, key=lambda x: x[1], reverse=True)[0]


    def best_results(self):
        return pd.DataFrame([(x[-1], x[0], x[1]) for x in self.best_model], 
                            columns=['model', 'params', 'valid']).sort_values(by=['valid'], ascending=False)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [6]:
class Finalize():
    def __init__(self, model):
        self.model = model

    def final_score(self, X_train, y_train, X_test, y_test):
        self.model.fit(X_train, y_train)
        return f'Accuracy of the final score is {accuracy_score(self.model.predict(X_test), y_test)}'

    def save_model(self, path):
        try:
            with open(path, "wb") as f:
                dump(self.model, f, protocol=-1)
            print(f'Successfully saved at {path}')
        except Exception as e:
            print(f'Error occur: {e}')

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])

preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
data = preprocessing.fit_transform(df)

X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest(data[1]).fit_transform(data[0])

svm = SVC
svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 
               'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)

dec_tree = DecisionTreeClassifier
dec_tree_parameters = {'max_depth': range(1, 50), 'class_weight': ('balanced', None), 
                       'criterion': ['entropy', 'gini'], 'random_state':[21]}
gs_tree = GridSearchCV(dec_tree, dec_tree_parameters, scoring='accuracy', n_jobs=-1)

rand_forest = RandomForestClassifier
rand_forest_parameters = {'n_estimators': (5, 10, 50, 100), 'max_depth': range(1, 50), 
                          'class_weight': ('balanced', None), 'criterion': ['entropy', 'gini'], 'random_state':[21]}
gs_rf = GridSearchCV(rand_forest, rand_forest_parameters, scoring='accuracy', n_jobs=-1)

grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {key: value for key, value in enumerate(('SVM', 'Decision Tree', 'Random Forest'))}

model_selection = ModelSelection(grids, grid_dict)
best_model = model_selection.choose()

Estimator: SVM


100%|██████████| 72/72 [06:42<00:00,  5.59s/it]


Best params {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878

Estimator: Decision Tree


100%|██████████| 196/196 [00:06<00:00, 29.95it/s]


Best params {'class_weight': None, 'criterion': 'gini', 'max_depth': 17, 'random_state': 21}
Best training accuracy: 0.8
Validation set accuracy score for best params: 0.87

Estimator: Random Forest


100%|██████████| 784/784 [02:53<00:00,  4.53it/s]

Best params {'class_weight': None, 'criterion': 'gini', 'max_depth': 27, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.857
Validation set accuracy score for best params: 0.893

Classifier with best validation set accuracty Random Forest


In [8]:
model_selection.best_results()

,model,params,valid
0,Random Forest,"{'class_weight': None, 'criterion': 'gini', 'm...",0.857
1,Decision Tree,"{'class_weight': None, 'criterion': 'gini', 'm...",0.800
2,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.773


In [9]:
final = Finalize(best_model[3].estimator(**best_model[0]))
final_score = final.final_score(X_train, y_train, X_test, y_test)
final_score

'Accuracy of the final score is 0.9171597633136095'

In [10]:
final_score_to_sav = round(float(final_score.rsplit(' ')[-1]), 3)
final_model_name = best_model[3].estimator.__name__
final.save_model(f'{final_model_name}_{final_score_to_sav}.sav')


Successfully saved at RandomForestClassifier_0.917.sav


In [11]:
with open(f'{final_model_name}_{final_score_to_sav}.sav', 'rb') as f:
    best_model_loaded = load(f)
    print(accuracy_score(best_model_loaded.predict(X_test), y_test))

0.9171597633136095
